# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [5]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [6]:
from transformers import AutoTokenizer, AutoModel

In [14]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized

In [8]:
class XMLRoBERTa():

    name = "xlm-roberta-large"

    def __init__(self):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.model.eval()

    def encode(
            self,
            inputs,
            strategy,
            chunk_max_size = 512,
            chunk_overlap = 64,
            **kwargs) -> torch.tensor:

        if strategy == "chunking":
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, inputs, chunk_max_size, chunk_overlap)
        if strategy == "first":
            tokenized = tokenize_first_startegy(self.tokenizer, inputs, chunk_max_size)

        with torch.inference_mode():

            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if strategy == "chunking":
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = [
                    t.mean(dim=1).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ]
            if strategy == "first":
                embeddings = outputs.last_hidden_state.mean(dim=1).detach().cpu()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings


class Qwen3_Embedding():

    name = "Qwen3-Embedding-0.6B"

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def encode(
            self,
            inputs,
            strategy = "chunking",
            chunk_max_size = 512,
            chunk_overlap = 64,
            **kwargs) -> torch.tensor:

        if strategy == "chunking":
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, inputs, chunk_max_size, chunk_overlap)
        if strategy == "first":
            tokenized = tokenize_first_startegy(self.tokenizer, inputs, chunk_max_size)

        with torch.inference_mode():

            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if strategy == "chunking":
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = [
                    self.__get_eos_token_embedding(t).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ]
            if strategy == "first":
                embeddings = self.__get_eos_token_embedding(outputs.last_hidden_state).detach().cpu()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

## LongEmbed LEMBWikimQARetrieval

In [9]:
from datasets import load_dataset

ds = load_dataset("dwzhu/LongEmbed", name="2wikimqa")
corpus = ds["corpus"]
queries = ds["queries"]
qrels = ds["qrels"]

In [10]:
qwen3_embed = Qwen3_Embedding()
xlm_roberta = XMLRoBERTa()

### Encoding document with each model with each text-preprocess strategy

In [11]:
# encode each document in a corpus
def encode_documents(corpus, model, batch_size, strategy, chunk_max_size, chunk_overlap):
    print(f"Encoding a coprus of length {len(corpus)}, processing using batches with size {batch_size}.")
    document_embeddings = {}
    for start in range(0, len(corpus), batch_size):
        end = start + batch_size
        batch = corpus[start:end]
        print(f"Processing the batch [{start+1}:{end}]")
        embedding = model.encode(batch["text"],
                                 strategy,
                                 chunk_max_size,
                                 chunk_overlap)

        for i, doc_id in enumerate(batch["doc_id"]):
            document_embeddings[doc_id] = embedding[i].numpy()

    return document_embeddings

# encode each query
def encode_queries(queries, model, batch_size, strategy, chunk_max_size, chunk_overlap):
    queries_embeddings = {}
    print("Encoding a set of queries.")
    for start in range(0, len(queries), batch_size):
        end = start + batch_size
        batch = queries[start:end]
        print(f"Processing the batch [{start+1}:{end}]")

        embedding = model.encode(batch["text"],
                                 "first")
        for i, q_id in enumerate(batch["qid"]):
            queries_embeddings[q_id] = embedding[i].numpy()

    return queries_embeddings

In [8]:
# encoding the queries with each mode
# strategy does not really matter in this case since the query will be one chunk long anyway
q3_query_embed = encode_queries(queries, qwen3_embed, 5, None, None, None)
roberta_query_embed = encode_queries(queries, xlm_roberta, 5, None, None, None)

Encoding a set of queries.
Processing the batch [1:5]
Processing the batch [6:10]
Processing the batch [11:15]
Processing the batch [16:20]
Processing the batch [21:25]
Processing the batch [26:30]
Processing the batch [31:35]
Processing the batch [36:40]
Processing the batch [41:45]
Processing the batch [46:50]
Processing the batch [51:55]
Processing the batch [56:60]
Processing the batch [61:65]
Processing the batch [66:70]
Processing the batch [71:75]
Processing the batch [76:80]
Processing the batch [81:85]
Processing the batch [86:90]
Processing the batch [91:95]
Processing the batch [96:100]
Processing the batch [101:105]
Processing the batch [106:110]
Processing the batch [111:115]
Processing the batch [116:120]
Processing the batch [121:125]
Processing the batch [126:130]
Processing the batch [131:135]
Processing the batch [136:140]
Processing the batch [141:145]
Processing the batch [146:150]
Processing the batch [151:155]
Processing the batch [156:160]
Processing the batch [1

In [9]:
pd.DataFrame(q3_query_embed).to_csv("./q3_query_embed.csv")
pd.DataFrame(roberta_query_embed).to_csv("./roberta_query_embed.csv")

In [10]:
del q3_query_embed, roberta_query_embed

In [11]:
q3_doc_embed_chunking_no_overlap = encode_documents(corpus, qwen3_embed, batch_size=1, strategy="chunking", chunk_max_size=512, chunk_overlap=0)
roberta_doc_embed_chunking_no_overlap = encode_documents(corpus, xlm_roberta, batch_size=1, strategy="chunking", chunk_max_size=512, chunk_overlap=0)

Encoding a coprus of length 300, processing using batches with size 1.
Processing the batch [1:1]
Processing the batch [2:2]
Processing the batch [3:3]
Processing the batch [4:4]
Processing the batch [5:5]
Processing the batch [6:6]
Processing the batch [7:7]
Processing the batch [8:8]
Processing the batch [9:9]
Processing the batch [10:10]
Processing the batch [11:11]
Processing the batch [12:12]
Processing the batch [13:13]
Processing the batch [14:14]
Processing the batch [15:15]
Processing the batch [16:16]
Processing the batch [17:17]
Processing the batch [18:18]
Processing the batch [19:19]
Processing the batch [20:20]
Processing the batch [21:21]
Processing the batch [22:22]
Processing the batch [23:23]
Processing the batch [24:24]
Processing the batch [25:25]
Processing the batch [26:26]
Processing the batch [27:27]
Processing the batch [28:28]
Processing the batch [29:29]
Processing the batch [30:30]
Processing the batch [31:31]
Processing the batch [32:32]
Processing the batc

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


Encoding a coprus of length 300, processing using batches with size 1.
Processing the batch [1:1]
Processing the batch [2:2]
Processing the batch [3:3]
Processing the batch [4:4]
Processing the batch [5:5]
Processing the batch [6:6]
Processing the batch [7:7]
Processing the batch [8:8]
Processing the batch [9:9]
Processing the batch [10:10]
Processing the batch [11:11]
Processing the batch [12:12]
Processing the batch [13:13]
Processing the batch [14:14]
Processing the batch [15:15]
Processing the batch [16:16]
Processing the batch [17:17]
Processing the batch [18:18]
Processing the batch [19:19]
Processing the batch [20:20]
Processing the batch [21:21]
Processing the batch [22:22]
Processing the batch [23:23]
Processing the batch [24:24]
Processing the batch [25:25]
Processing the batch [26:26]
Processing the batch [27:27]
Processing the batch [28:28]
Processing the batch [29:29]
Processing the batch [30:30]
Processing the batch [31:31]
Processing the batch [32:32]
Processing the batc

In [14]:
pd.DataFrame(q3_doc_embed_chunking_no_overlap).to_csv("q3_doc_embed_chunking_no_overlap.csv")
pd.DataFrame(roberta_doc_embed_chunking_no_overlap).to_csv("roberta_doc_embed_chunking_no_overlap.csv")

In [16]:
q3_doc_embed_first = encode_documents(corpus, qwen3_embed, batch_size=5, strategy="first", chunk_max_size=512, chunk_overlap=None)
roberta_doc_embed_first = encode_documents(corpus, xlm_roberta, batch_size=5, strategy="first", chunk_max_size=512, chunk_overlap=None)

Encoding a coprus of length 300, processing using batches with size 5.
Processing the batch [1:5]
Processing the batch [6:10]
Processing the batch [11:15]
Processing the batch [16:20]
Processing the batch [21:25]
Processing the batch [26:30]
Processing the batch [31:35]
Processing the batch [36:40]
Processing the batch [41:45]
Processing the batch [46:50]
Processing the batch [51:55]
Processing the batch [56:60]
Processing the batch [61:65]
Processing the batch [66:70]
Processing the batch [71:75]
Processing the batch [76:80]
Processing the batch [81:85]
Processing the batch [86:90]
Processing the batch [91:95]
Processing the batch [96:100]
Processing the batch [101:105]
Processing the batch [106:110]
Processing the batch [111:115]
Processing the batch [116:120]
Processing the batch [121:125]
Processing the batch [126:130]
Processing the batch [131:135]
Processing the batch [136:140]
Processing the batch [141:145]
Processing the batch [146:150]
Processing the batch [151:155]
Processing

In [17]:
pd.DataFrame(q3_doc_embed_first).to_csv("q3_doc_embed_first.csv")
pd.DataFrame(roberta_doc_embed_first).to_csv("roberta_doc_embed_first.csv")

In [18]:
q3_doc_embed_chunking_64_overlap = encode_documents(corpus, qwen3_embed, batch_size=1, strategy="chunking", chunk_max_size=512, chunk_overlap=64)
roberta_doc_embed_chunking_64_overlap = encode_documents(corpus, xlm_roberta, batch_size=1, strategy="chunking", chunk_max_size=512, chunk_overlap=64)

Encoding a coprus of length 300, processing using batches with size 1.
Processing the batch [1:1]
Processing the batch [2:2]
Processing the batch [3:3]
Processing the batch [4:4]
Processing the batch [5:5]
Processing the batch [6:6]
Processing the batch [7:7]
Processing the batch [8:8]
Processing the batch [9:9]
Processing the batch [10:10]
Processing the batch [11:11]
Processing the batch [12:12]
Processing the batch [13:13]
Processing the batch [14:14]
Processing the batch [15:15]
Processing the batch [16:16]
Processing the batch [17:17]
Processing the batch [18:18]
Processing the batch [19:19]
Processing the batch [20:20]
Processing the batch [21:21]
Processing the batch [22:22]
Processing the batch [23:23]
Processing the batch [24:24]
Processing the batch [25:25]
Processing the batch [26:26]
Processing the batch [27:27]
Processing the batch [28:28]
Processing the batch [29:29]
Processing the batch [30:30]
Processing the batch [31:31]
Processing the batch [32:32]
Processing the batc

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


Encoding a coprus of length 300, processing using batches with size 1.
Processing the batch [1:1]
Processing the batch [2:2]
Processing the batch [3:3]
Processing the batch [4:4]
Processing the batch [5:5]
Processing the batch [6:6]
Processing the batch [7:7]
Processing the batch [8:8]
Processing the batch [9:9]
Processing the batch [10:10]
Processing the batch [11:11]
Processing the batch [12:12]
Processing the batch [13:13]
Processing the batch [14:14]
Processing the batch [15:15]
Processing the batch [16:16]
Processing the batch [17:17]
Processing the batch [18:18]
Processing the batch [19:19]
Processing the batch [20:20]
Processing the batch [21:21]
Processing the batch [22:22]
Processing the batch [23:23]
Processing the batch [24:24]
Processing the batch [25:25]
Processing the batch [26:26]
Processing the batch [27:27]
Processing the batch [28:28]
Processing the batch [29:29]
Processing the batch [30:30]
Processing the batch [31:31]
Processing the batch [32:32]
Processing the batc

In [20]:
pd.DataFrame(q3_doc_embed_chunking_64_overlap).to_csv("q3_doc_embed_chunking_64_overlap.csv")
pd.DataFrame(roberta_doc_embed_chunking_64_overlap).to_csv("roberta_doc_embed_chunking_64_overlap.csv")

### Calculating evaluation metrics

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [22]:
def MAP_at_K(documents_embed, queries_embed, qrel, k):
    sum_ap_at_k = 0
    n_queries = 0
    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            ap_at_k = 1 / true_document_position
        except ValueError:
            ap_at_k = 0
        sum_ap_at_k += ap_at_k
        n_queries += 1
    map = sum_ap_at_k / n_queries
    return map

In [23]:
def mean_nDCG_at_k(documents_embed, queries_embed, qrel, k):
    sum_ndcg_at_k = 0
    n_queries = 0

    idcg_at_k = 1
    for i in range(1, 10+1):
        idcg_at_k += 1 / np.log2(i + 2)

    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            dcg_at_k = 1 / np.log2(true_document_position + 1)
        except ValueError:
            dcg_at_k = 0
        ndcg_at_k = dcg_at_k / idcg_at_k
        sum_ndcg_at_k += ndcg_at_k
        n_queries += 1

    mean_ndcg = sum_ndcg_at_k / n_queries
    return mean_ndcg

In [ ]:
roberta_doc_embeddings = pd.read_csv("./doc_embeddings/roberta_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_doc_embeddings = pd.read_csv("./doc_embeddings/q3_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

roberta_query_embeddings = pd.read_csv("./query_embeddings/roberta_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_query_embeddings = pd.read_csv("./query_embeddings/q3_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

In [ ]:
print("XLM-RoBERTa-large nDCG@k: ", mean_nDCG_at_k(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))
print("XLM-RoBERTa-large MAP@k: ", MAP_at_K(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))

XLM-RoBERTa-large nDCG@k:  0.027452690239719697
XLM-RoBERTa-large MAP@k:  0.10360449735449732


In [ ]:
print("Qwen3 Embedding nDCG@k: ", mean_nDCG_at_k(q3_doc_embeddings, q3_query_embeddings, qrels, 10))
print("Qwen3 Embedding MAP@k: ", MAP_at_K(q3_doc_embeddings, q3_query_embeddings, qrels, 10))

Qwen3 Embedding nDCG@k:  0.15436043151119166
Qwen3 Embedding MAP@k:  0.7148743386243387
